# Categorize Responses in the Behavioural Set

In [ ]:
import sys, os

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(),".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0,PARENT_DIR)

sys.path

In [ ]:
import torch
import json
from typing import List, Literal, Optional, Union, Dict
from enum import Enum
from collections import Counter
import datetime
from dataclasses import dataclass
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Local imports
from config import settings
from gcp_utils import download_from_gcs
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans
from error_analysis.error_categorization import ErrorCategorizer
from error_analysis.error_taxonomy import BehaviouralExample

VERSION = "v02"
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 2
INFERENCE_PIPELINE_VERSION = "v01"

In [ ]:
## Load Required Data Files

# Load id2label mapping
print("📂 Loading id2label mapping...")
with open("../v01/data/id2label.json", "r") as f:
    id2label = json.load(f)

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}

print(f"✅ Loaded {len(id2label)} label mappings")
print(f"Labels: {id2label}")

# Load behavioural evaluation set
print("\n📂 Loading behavioural evaluation set...")
with open("../v01/behavioural_set.json", "r") as f:
    behavioural_set = json.load(f)

print(f"✅ Loaded {len(behavioural_set)} categories")
print(f"Categories: {list(behavioural_set.keys())}")

# Count total examples
total_examples = sum(len(examples) for examples in behavioural_set.values())
print(f"Total test examples: {total_examples}")

In [ ]:
# Load model from the local directory
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()
print("✅ Model is ready to be used")

## Load the Behavioural Set

In [ ]:
# Fit the behavioural set into the dataclasses
for k,examples in behavioural_set.items():
    for i,ex in enumerate(examples):
        behavioural_set[k][i] = BehaviouralExample(**ex)
behavioural_set


## Instantiate the Error Categorizer

In [ ]:
error_categorizer = ErrorCategorizer()

## Single Sample Example

In [ ]:
print("Working example:")
my_example = behavioural_set['Long, realistic clinical sentences (THIS IS GOLD 🥇)'][0]
text = my_example.example
print(f"\t{text}")
tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
spans = word_labels_to_spans(text=text, word_offsets=word_offsets, word_labels=word_labels)
print("SPANS:")
for s in spans:
    print(f"\t{s}")

*Snipped to check if the expected entities are in the returned spans*

In [ ]:
my_example.entities_with_labels

In [ ]:
# Example to show how error_categorizer._is_entity_in_spans(a['ent'], spans) works
sample_entity_data = my_example.entities_with_labels[0]
ent = sample_entity_data['ent']
idx = error_categorizer._is_entity_in_spans(ent, spans)
print(f"The entity {ent} was found in the following span: {spans[idx]}")
# See which entities from the behavioural examples are in:
# For present entities, we also track which span index they were found in
present_with_indices = [(e, error_categorizer._is_entity_in_spans(e["ent"], spans)) for e in my_example.entities_with_labels]
present = [(e,idx) for e, idx in present_with_indices if idx is not None]
missing = [(e,idx) for e, idx in present_with_indices if idx is None]

print(f"present_with_indices:\n\t{present_with_indices}")
print(f"present:\n\t{present}")
print(f"missing:\n\t{missing}")

*Analyze the present errors for this single example*

In [ ]:
errors = error_categorizer._check_entity_detection(
    example=my_example,
    spans=spans
)

present_errors = errors.get("present_errors", [])
missing_entities = errors.get("missing_entities", [])
false_positives = errors.get("false_positives", [])

print("============== ERRORS FOUND ==============")

if present_errors:
    print("\n--- Present errors (detected but wrong boundary/label) ---")
    for p in present_errors:
        e = p['entity']   # ground truth
        s = p['span']     # prediction
        print(f"  Ground Truth : {e['ent']} ({e['label']}) [{e['start']}:{e['end']}]")
        print(f"  Prediction   : {s['text']} ({s['label']}) [{s['start']}:{s['end']}]")
        print(f"  Error(s)     : {p['errors']}\n")

if missing_entities:
    print("\n--- Missing entities (not detected at all) ---")
    for p in missing_entities:
        e = p['entity']   # ground truth only — no predicted span
        print(f"  Ground Truth : {e['ent']} ({e['label']}) [{e['start']}:{e['end']}]")
        print(f"  Prediction   : (not detected)")
        print(f"  Error(s)     : {p['errors']}\n")

if false_positives:
    print("\n--- False positives (predicted but not expected) ---")
    for p in false_positives:
        s = p['span']     # prediction only — no matching ground truth
        print(f"  Prediction   : {s['text']} ({s['label']}) [{s['start']}:{s['end']}]")
        print(f"  Ground Truth : (none — unexpected prediction)")
        print(f"  Error(s)     : {p['errors']}\n")

if not any([present_errors, missing_entities, false_positives]):
    print("  No errors found.")

# Run Error Categorization thoughout the entire dataset

In [ ]:
def _run_through_behavioural_set(
    error_categorizer: ErrorCategorizer,
    behavioural_set: Dict[str, List[BehaviouralExample]],
    model,
    tokenizer,
    id2label: dict,
    device: str,
) -> dict:
    """Run inference + `_check_entity_detection` on every example.

    Clears `error_categorizer.error_counts` so one full sweep has a single aggregate counter.
    Returns dict with `error_counts` (Counter), plus flat lists of error records for inspection.
    """
    error_categorizer.error_counts.clear()

    total_present_errors: List[dict] = []
    total_missing_entities: List[dict] = []
    total_false_positives: List[dict] = []

    for ex_type, examples in behavioural_set.items():
        print(f"Category: {ex_type}")
        for ex in examples:
            text = ex.example
            tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
                text=text,
                model=model,
                tokenizer=tokenizer,
                id2label=id2label,
                device=device,
            )
            spans = word_labels_to_spans(
                text=text,
                word_offsets=word_offsets,
                word_labels=word_labels,
            )
            errors = error_categorizer._check_entity_detection(example=ex, spans=spans)
            pe = errors.get("present_errors", [])
            me = errors.get("missing_entities", [])
            fp = errors.get("false_positives", [])
            total_present_errors.extend(pe)
            total_missing_entities.extend(me)
            total_false_positives.extend(fp)
        print(f"  Processed {len(examples)} examples\n")

    return {
        "error_counts": error_categorizer.error_counts,
        "present_errors": total_present_errors,
        "missing_entities": total_missing_entities,
        "false_positives": total_false_positives,
    }


In [ ]:
results = _run_through_behavioural_set(
    behavioural_set=behavioural_set,
    error_categorizer=error_categorizer,
    model=model,
    id2label=id2label,
    tokenizer=tokenizer,
    device=device
)

In [ ]:
# These results are very important they dictate the next steps for improving the model!
results["error_counts"]

In [ ]:
{'entity': {'ent': 'bradypnea',
   'label': 'SYMPTOM_POS',
   'start': 13,
   'end': 22},
  'span': {'start': 13,
   'end': 38,
   'text': 'bradypnea since yesterday',
   'label': 'SYMPTOM_POS'},
  'span_idx': 2,
  'errors': ['Boundary Overreach']}


{'entity': {'ent': 'hepatic dysfunction',
   'label': 'O',
   'start': 45,
   'end': 64},
  'span': {'start': 45,
   'end': 64,
   'text': 'hepatic dysfunction',
   'label': 'SYMPTOM_POS'},
  'span_idx': 4,
  'errors': ['Irrelevant Span Mislabeling']}


{'entity': None,
  'span': {'start': 11,
   'end': 21,
   'text': 'complaints',
   'label': 'SYMPTOM_NEG'},
  'span_idx': 2,
  'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
  'reasoning': "Predicted entity span 'complaints' not in expected entities"},
 {'entity': None,
  'span': {'start': 10,
   'end': 23,
   'text': 'questionnaire',
   'label': 'SYMPTOM_POS'},
  'span_idx': 1,
  'errors': ['Irrelevant Span Mislabeling', 'Model Overgeneralization'],
  'reasoning': "Predicted entity span 'questionnaire' not in expected entities"}



{'entity': {'ent': 'wheezing',
    'label': 'SYMPTOM_POS',
    'start': 30,
    'end': 38},
   'span': {'start': 30,
    'end': 49,
    'text': 'wheezing is related',
    'label': 'SYMPTOM_NEG'},
   'span_idx': 5,
   'errors': ['Incorrect Polarity Assignment', 'Boundary Overreach']}


   {'entity': {'ent': 'chills',
    'label': 'SYMPTOM_NEG',
    'start': 148,
    'end': 154},
   'span': {'start': 148,
    'end': 154,
    'text': 'chills',
    'label': 'CONFLICT-I-SYMPTOM_NEG-I-SYMPTOM_POS'},
   'span_idx': 9,
   'errors': ['Tokenization Artifacts', 'BIO Sequencing Errors']}

In [ ]:
results['missing_entities']

How come the results were better when I tried the entire network...?

In [ ]:
results